# Lesson 1: Aura Graph Analytics Workflows

This notebook is a companion to [Introduction to Aura Graph Analytics on Graph Academy](https://graphacademy.neo4j.com/courses/workshop-gds?ref=workshop-notebooks).

**Duration:** ~10 minutes  
**Module:** Aura Graph Analytics  
**Dataset:** Cargo 2000 Freight Forwarding

## What You'll Learn

- How to create a GDS Session for pathfinding analysis
- How to load and project the logistics network
- How to run Dijkstra and Yen's algorithms to find optimal routes
- **How to visualize the graph using neo4j_viz**

## Prerequisites

- Completion of Lessons 1-2 (AGA fundamentals)
- AuraDB instance with Cargo 2000 data loaded
- Aura API credentials

## The Cargo 2000 Dataset

The Cargo 2000 dataset represents real air cargo logistics from IATA:

**Nodes:**
- `EntryPoint` — Source airports where freight originates
- `Destination` — Delivery airports
- `DepartureWarehouse`, `ArrivalWarehouse` — Storage facilities
- `TransferPoint` — Consolidation hubs

**Relationships:**
- `RECEPTION` — Freight received at warehouse
- `DEPARTURE` — Freight leaves warehouse
- `TRANSPORT` — Movement between locations
- `DELIVERY` — Final delivery to destination

Each relationship has an `effectiveMinutes` property—the actual time for that process step.

In this notebook, we'll:

1. **Connect to Aura Graph Analytics** - Set up our ephemeral compute environment
2. **Explore the Cargo 2000 dataset** - Understand the freight forwarding network
3. **Learn the projection workflow** - See how data moves from AuraDB → Session → Projected Graph

By the end, you'll understand the AGA workflow well enough to run pathfinding 
algorithms in the next lesson.

**The Big Picture:** We have 5 months of shipping data. Some routes we've used 
historically may not be optimal. Before we can find better routes, we need to 
understand our data and how to work with it in GDS.

## Part 1: Create GDS Session

In [ ]:
# Imports and environment variables
import os
import pandas as pd
from datetime import timedelta
from IPython.display import display
from graphdatascience.session import GdsSessions, AuraAPICredentials
from graphdatascience.session import DbmsConnectionInfo, SessionMemory
from dotenv import load_dotenv

# Import neo4j driver Result for graph transformations
from neo4j import Result

# Import neo4j_viz for visualization
from neo4j_viz.gds import from_gds 
from neo4j_viz.neo4j import from_neo4j
from neo4j_viz import Layout
from neo4j_viz.colors import ColorSpace

# Load environment variables
load_dotenv()

# Get Aura API credentials
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id = os.getenv('AURA_PROJECT_ID')  # set in .env only if your Aura account has multiple projects

# Get AuraDB connection info
uri = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

In [ ]:
# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id)
)

In [ ]:
# Create a GDS Session
gds = sessions.get_or_create(
    session_name="logistics-pathfinding",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)
print(f"Client ID starts with: {client_id[:12]}...")

# Verify connection
gds.verify_connectivity()
print(f"Connected to GDS Session: logistics-pathfinding")

## Helper functions

The following functions just create a Neo4j Driver and a neo4j-viz function so that we can get graph visualizations right inside the notebook. They are a nicety -- not necessary for the core of graph data science. 

In [ ]:
# Helper function: Execute query with Neo4j driver (for visualizations)
# Uses the global uri, username, password by default
def execute_neo4j_query(query, _uri=None, _username=None, _password=None, database=os.getenv("AURA_DATABASE") or "neo4j"):
    from neo4j import GraphDatabase, RoutingControl
    
    # Use global credentials if not provided
    conn_uri = _uri if _uri is not None else uri
    conn_username = _username if _username is not None else username
    conn_password = _password if _password is not None else password
    
    with GraphDatabase.driver(conn_uri, auth=(conn_username, conn_password)) as driver:
        driver.verify_connectivity()
        result = driver.execute_query(
            query,
            database_=database,
            routing_=RoutingControl.READ,
            result_transformer_=Result.graph,
        )
    return result

    # Helper function: Create visualization from Cypher query
# Just pass the query - credentials are pulled from global scope automatically!
def visualize_query(query, database=os.getenv("AURA_DATABASE") or "neo4j", color_field=None, _uri=None, _username=None, _password=None):
    result = execute_neo4j_query(query, _uri, _username, _password, database)
    VG = from_neo4j(result)
    
    if color_field:
        VG.color_nodes(field=color_field)
    
    return VG

# Helper function: Visualize a GDS graph projection
def visualize_projection(G):
    return from_gds(gds, G, max_node_count=20)

print("Helper functions loaded: execute_neo4j_query(), visualize_query(), visualize_projection()")

## Part 2: Explore the Data

Before projecting, let's understand the logistics network.

The Cargo 2000 dataset has **two layers**:

**1. Raw Operations Layer** (what we'll use for pathfinding):
- Nodes: EntryPoint, DepartureWarehouse, TransferPoint, ArrivalWarehouse, Destination
- Relationships: RECEPTION, DEPARTURE, TRANSPORT, DELIVERY
- Each relationship has `effectiveMinutes` - actual time for that step

**2. Historical Summary Layer** (what we'll compare against):
- HistoricalRoute nodes connect EntryPoints to Destinations
- These represent routes we've actually used in the past
- We'll compare these against optimal routes later

For now, let's project the Historical Routes to see which origin-destination 
pairs exist in our data.

In [ ]:
# Visualize the database schema - simple one-liner!
print("Cargo 2000 Logistics Network Schema")
print("=" * 50)

VG = visualize_query("CALL db.schema.visualization()", color_field="caption")
VG.render()

In [ ]:
# Project an undirected graph
G_historical_route, result = gds.graph.project(
    "example-graph",    #Title of the graph
    """
    CALL {
      MATCH (a1:EntryPoint)-[r:HAS_HISTORICAL_ROUTE]->(:HistoricalRoute)-[:TERMINATES_AT]->(a2:Destination) //Match query
      WHERE a1 <> a2
      RETURN a1 AS source, 
            a2 AS target,                                                 //Declare nodes
            type(r) AS relType,                                           //Gather rels
            r.effectiveMinutes AS effectiveMinutes,                       //Gather properties
            count(r) AS relationshipCount                                 //Aggregate relationships
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      relationshipType: relType,
      relationshipProperties: {effectiveMinutes: effectiveMinutes, relationshipCount: relationshipCount}
    })
    """,
    undirected_relationship_types=["*"] # Undirected relationships are specified outside the CALL block
)

print(f"Projected graph: {G_historical_route.name()}")
print(f"  Nodes: {G_historical_route.node_count():,}")
print(f"  Relationships: {G_historical_route.relationship_count():,}")

The following cell references our neo4j-viz[gds] function and shows us a sample of the projected graph.

In [ ]:
VG = visualize_projection(G_historical_route)
VG.render()

List active sessions with `sessions.list()`

In [ ]:
sessions.list()

Your projection now 'lives' inside the session. You can now reference the graph in the session using the same commands you're used to.

In [ ]:
gds.graph.list()

If we drop the graph...

In [ ]:
gds.graph.drop(G_historical_route)

The session still exists...

In [ ]:
sessions.list()

We can project another graph into it:

### Native projection vs Cypher projection (new in Aura Graph Analytics)

Native projections read node labels and relationship types directly from database storage — the fastest way to load a graph, with no Cypher query. The Cypher projection that follows builds the same airport graph using a query instead.

Because this airport graph needs no aggregation or computed properties, native and Cypher produce an identical result here — native is simply faster. (When you need computed or aggregated relationships, use the Cypher projection.)

In [ ]:
# Native projection - the same Airport / SENDS_TO graph, read straight from storage.
G_native, _ = gds.v2.graph.project_native(
    "example-graph-native",
    ["Airport"],
    ["SENDS_TO"],
    relationship_properties=["flightCount"],
    undirected_relationship_types=["SENDS_TO"],
)
print(f"Native projection: {G_native.node_count():,} nodes, {G_native.relationship_count():,} relationships")
G_native.drop()  # comparison only - the cells below use the Cypher projection

In [ ]:
G, result = gds.graph.project(
    "example-graph-cypher",    #Title of the graph
    """
    CALL {
      MATCH (a1:Airport)-[r:SENDS_TO]->(a2:Airport) //Match query
      RETURN a1 AS source,
            a2 AS target,
            type(r) AS relType,
            r.flightCount AS flightCount
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      relationshipType: relType,
      relationshipProperties: {flightCount: flightCount}
    })
    """,
    undirected_relationship_types=["*"] # Undirected relationships are specified outside the CALL block
)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

In [ ]:
VG = visualize_projection(G)
VG.render()

To run an algorithm, we reference `G` (our projected graph) in the algorithm call.

Here's a quick PageRank example - this finds which airports are most "central" 
to our network. We won't dive deep into PageRank now (that's in the Centrality module), 
but this demonstrates the basic pattern:

1. Project a graph → `G`
2. Run algorithm on `G` → results

In [ ]:
print("Running PageRank ...")
df = gds.pageRank.stream(G)
df

If we leave the session running -- even with no projection -- it's still costing money.

To delete the session:

In [ ]:
gds.delete()
# or
# sessions.delete(session_name="my-session")

Now let's run a query:

In [ ]:
# Count nodes and relationships
df = gds.run_cypher("""
    MATCH (n)
    WITH count(n) AS node_count
    MATCH ()-[r]->()
    RETURN node_count AS nodes, count(r) AS relationships
""")
print("Graph size:")
display(df)

## Why This Failed

The query failed because we deleted our GDS session. 

The `gds` object is your connection to the session. 

When you call `gds.delete()`, the session is terminated and `gds` becomes invalid.

To continue working, you have two options:
1. Create a new session (what we'll do below)
2. Use a standard Neo4j driver for direct database queries (bypasses GDS entirely)

**Sessions are ephemeral**. When they're gone, so is any in-memory projection data. 

Always write important results back to the database before deleting.

In [ ]:
gds = sessions.get_or_create(
    session_name="new-session",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)
print(f"Client ID starts with: {client_id[:12]}...")

# Verify connection
gds.verify_connectivity()
print(f"Connected to GDS Session: new-session")

Now, let's explore the data:

In [ ]:
# Count nodes and relationships
df = gds.run_cypher("""
    MATCH (n)
    WITH count(n) AS node_count
    MATCH ()-[r]->()
    RETURN node_count AS nodes, count(r) AS relationships
""")
print("Graph size:")
display(df)

In [ ]:
# View node types and counts
df = gds.run_cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY count DESC
""")
print("Node types:")
display(df)

In [ ]:
# View relationship types and counts
df = gds.run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS type, count(*) AS count, 
           round(avg(r.effectiveMinutes), 2) AS avg_minutes
    ORDER BY count DESC
""")
print("Relationship types with average transit times:")
display(df)

In [ ]:
VG = visualize_query("""
    MATCH path = (e:EntryPoint)-[:RECEPTION|DEPARTURE|TRANSPORT|DELIVERY*1..6]->(d:Destination)
    RETURN path
    LIMIT 10
    """)
VG.render()

In [ ]:
sessions.delete(session_name="new-session")
# or
# gds.delete() 

In the next section, we'll explore Dijkstra's and how it works on our familiar Movies dataset.

## Summary

You've explored the dataset we'll be using throughout this session. 

You've learned the Aura Graph Analytics workflow:

* Create sessions with `sessions.get_or_create()`
* Project graphs with `gds.graph.project()` and `gds.graph.project.remote()`
* Run algorithms on projected graphs
* Manage sessions (list, delete)

Now that you understand the workflow, we'll learn **Dijkstra's algorithm** - the foundation for finding optimal shipping routes. 

We'll practice on the familiar Movies dataset first, then apply it to our logistics network.